# Volve Field - Notebook 1: Cleaning the Data
This notebook does in Python exactly whats in Excel:
load the raw production data, tidy it up, flag each well as a producer or
injector, and build a few useful columns (KPIs).

**Data:** Equinor Volve open dataset (CC BY-NC-SA 4.0) — Daily Production sheet.

The daily sheet is used because it has more detail (pressure, choke) than the monthly sheet, which is better suited to Excel dashboards.

In [17]:
import pandas as pd
import numpy as np

df = pd.read_excel("../data/Volve_Production_Data.xlsx", sheet_name="Daily Production Data") # define the dataframe df

df.head() # show first 5 rows

df.shape # (number of rows, number of columns)


(15634, 24)

In [18]:
# Rename the columns to plain simple names 
# `rename` takes a dictionary: "old name": "new name"

df = df.rename(columns={
    "DATEPRD": "Date",
    "WELL_BORE_CODE": "Well",
    "ON_STREAM_HRS": "On Stream Hours",
    "BORE_OIL_VOL": "Oil",
    "BORE_GAS_VOL": "Gas",
    "BORE_WAT_VOL": "Water",
    "BORE_WI_VOL": "Water Injected",
    "FLOW_KIND": "Flow Kind",
    "WELL_TYPE": "Well Type",
    "AVG_DOWNHOLE_PRESSURE": "Downhole Pressure",
    "AVG_CHOKE_SIZE_P": "Choke Size",
    "AVG_WHP_P": "Wellhead Pressure"
})

df.head()

,Date,Well,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,On Stream Hours,Downhole Pressure,...,AVG_CHOKE_UOM,Wellhead Pressure,AVG_WHT_P,DP_CHOKE_SIZE,Oil,Gas,Water,Water Injected,Flow Kind,Well Type
0,2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,WI
1,2014-04-08,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
2,2014-04-09,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
3,2014-04-10,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
4,2014-04-11,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,...,%,33.09788,10.47992,33.07195,0.0,0.0,0.0,NaN,production,OP


In [19]:
# Flag each row as Producer or Injector

# set every row to "Producer". Then, for the rows where `Flow Kind` is
# "injection", change it to "Injector". add column well role to the dataframe

df["Well Role"] = "Producer"
df.loc[df["Flow Kind"] == "injection", "Well Role"] = "Injector"

df["Well Role"].value_counts()

#df.head()


Well Role
Producer    9161
Injector    6473
Name: count, dtype: int64

In [20]:
# injector rows had "NULL" in the oil/gas/water columns, replaced with 0. affects averages later hence "On Stream Hours" used to filter

df["Oil"] = df["Oil"].fillna(0)
df["Gas"] = df["Gas"].fillna(0)
df["Water"] = df["Water"].fillna(0)
df["Water Injected"] = df["Water Injected"].fillna(0)

df[["Oil", "Gas", "Water", "Water Injected"]].isnull().sum()

Oil               0
Gas               0
Water             0
Water Injected    0
dtype: int64

In [21]:
# Build the KPI columns as same calc in excel. Making a new column in pandas is just: `df["New Column"] = a calculation`
# 
# Oil Rate (Sm³/day) = oil per producing day. 
# On Stream Hours / 24 converts hours to days, matching the daily rates NPD reports.

df["Oil Rate"] = df["Oil"] / (df["On Stream Hours"] / 24) # Oil Rate (Sm³/day)

df["Water Cut"] = df["Water"] / (df["Oil"] + df["Water"]) # Water Cut (fraction/percentage)

df["GOR"] = df["Gas"] / df["Oil"] # GOR (Gas-Oil Ratio) = gas produced per unit of oil.

df = df.replace([np.inf, -np.inf], np.nan) # np.nan is pd word for "blank").

df[["Date", "Well", "Well Role", "Oil", "Oil Rate", "Water Cut", "GOR"]].head()

#df.head() # check with top excel row in data sheet


,Date,Well,Well Role,Oil,Oil Rate,Water Cut,GOR
0,2014-04-07,NO 15/9-F-1 C,Producer,0.0,NaN,NaN,NaN
1,2014-04-08,NO 15/9-F-1 C,Producer,0.0,NaN,NaN,NaN
2,2014-04-09,NO 15/9-F-1 C,Producer,0.0,NaN,NaN,NaN
3,2014-04-10,NO 15/9-F-1 C,Producer,0.0,NaN,NaN,NaN
4,2014-04-11,NO 15/9-F-1 C,Producer,0.0,NaN,NaN,NaN


In [22]:
df.to_csv("../data/volve_daily_clean.csv", index=False)
print("Saved. Rows:", df.shape[0], "Columns:", df.shape[1])

Saved. Rows: 15634 Columns: 28


In [23]:
#import os
#print(os.getcwd())